In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline


from sklearn.preprocessing import StandardScaler

from sklearn.impute import SimpleImputer

                                                      #nueral network architecture
class Breast_Cancer_Identification_Model(nn.Module):
  def __init__(self, num_inputs = 30, h1 = 891, h2 = 891, h3 = 891, h4 = 297,
                                      h5 = 297, h6 = 297, h7 = 99, h8 = 99,
                                      h9 = 99, h10 = 33, h11 = 33, h12 = 33,
                                                h13= 11, num_outputs = 2):

    super().__init__()
#mapping nueral network
    self.fc1 = nn.Linear(num_inputs, h1)

    self.fc2 = nn.Linear(h1, h2)
    self.fc3 = nn.Linear(h2, h3)
    self.fc4 = nn.Linear(h3, h4)
    self.fc5 = nn.Linear(h4, h5)
    self.fc6 = nn.Linear(h5, h6)
    self.fc7 = nn.Linear(h6, h7)
    self.fc8 = nn.Linear(h7, h8)
    self.fc9 = nn.Linear(h8, h9)
    self.fc10 = nn.Linear(h9, h10)
    self.fc11 = nn.Linear(h10, h11)
    self.fc12 = nn.Linear(h11, h12)
    self.fc13 = nn.Linear(h12, h13)



    self.fc_out = nn.Linear(h13, num_outputs)

  def forward(self, x):

# using relu(rectified linear unit) as the activation and backpropagation
    x = F.relu(self.fc1(x))

    x = F.relu(self.fc2(x))

    x = F.relu(self.fc3(x))

    x = F.relu(self.fc4(x))

    x = F.relu(self.fc5(x))

    x = F.relu(self.fc6(x))

    x = F.relu(self.fc7(x))

    x = F.relu(self.fc8(x))

    x = F.relu(self.fc9(x))
    x = F.relu(self.fc10(x))
    x = F.relu(self.fc11(x))
    x = F.relu(self.fc12(x))
    x = F.relu(self.fc13(x))



    x = self.fc_out(x)

    return x


#loading in data
data_url = '/content/data.csv'
data = pd.read_csv(data_url)

data['diagnosis'] = data['diagnosis'].map({'M': 1, 'B': 0})

#diagnosis is the target column where its trained to predict
X = data.drop(['diagnosis', 'id', 'Unnamed: 32'], axis=1)
y = data['diagnosis']



if X.isnull().sum().sum() > 0:
    print(f"Found {X.isnull().sum().sum()} NaN values in features. Imputing with mean strategy.")
    imputer = SimpleImputer(strategy='mean')
    X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns, index=X.index)


from sklearn.model_selection import train_test_split


 #learning with using torch
X_train_df, X_test_df, y_train_series, y_test_series = train_test_split(X, y, test_size=.4, random_state=167)
original_test_indices = X_test_df.index


X_train_df.reset_index(drop=True, inplace=True)


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_df)
X_test_scaled = scaler.transform(X_test_df)


X_train = torch.FloatTensor(X_train_scaled)
X_test = torch.FloatTensor(X_test_scaled)

y_train = torch.LongTensor(y_train_series.values)
y_test = torch.LongTensor(y_test_series.values)


class_counts = y_train_series.value_counts()
class_weights = torch.FloatTensor([1.0 / class_counts[0], 1.0 / class_counts[1]])

criterion = nn.CrossEntropyLoss(weight=class_weights)


torch.manual_seed(142)

model = Breast_Cancer_Identification_Model(num_inputs=30, num_outputs=2)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0005, betas=(0.9, 0.999),
                                     eps=1e-08, weight_decay=0.0001, amsgrad=False,
                                     foreach=None, maximize=False, capturable=None,
                                     differentiable=False, fused=False,
                                     decoupled_weight_decay=True)
# how many times the nn goes through the data
epochs = 10000
losses = []

for i in range(epochs):
  y_pred = model.forward(X_train)

  cost = criterion(y_pred, y_train)
  losses.append(cost.detach().numpy())


  if i % 10 == 0:
    print(f'Epoch: {i} Cost: {cost.item()}')

  optimizer.zero_grad()

  cost.backward()

  optimizer.step()

# for graphing the loss function to see how its converging
plt.plot(range(epochs), losses)
plt.ylabel('Cost')
plt.xlabel('Epoch')
plt.title('Cost over Epochs with Best Hyperparameters')
plt.show()


correct = 0
misclassified_samples = []

with torch.no_grad():
  for i, data_sample in enumerate(X_test):
    y_val = model.forward(data_sample)
    predicted_class = y_val.argmax().item()
    actual_class = y_test[i].item()

#shows how accurate it is
    if predicted_class == actual_class:
      correct +=1
    else:
      misclassified_samples.append({
          'test_index_in_X_test': i,
          'original_data_index': original_test_indices[i],
          'actual_class': actual_class,
          'predicted_class': predicted_class,
          'prediction_output': y_val.tolist() \
      })

print(f'                  Breast Cancer Neural Network went {correct}/{len(X_test)} correct!!!!!')
#shows what was wrong if so. this can be used to help improve the model by finding patterns within the wrong prediction if present.
print("\n--- Misclassified Samples Details ---")
if len(misclassified_samples) > 0:
    for sample_info in misclassified_samples:
        original_idx = sample_info['original_data_index']
        actual = sample_info['actual_class']
        predicted = sample_info['predicted_class']
        prediction_output = sample_info['prediction_output']

        print(f"\nOriginal Data Index: {original_idx}")
        print(f"Actual Diagnosis (0=B, 1=M): {actual}")
        print(f"Predicted Diagnosis (0=B, 1=M): {predicted}")
        print(f"Model Raw Output: {prediction_output}")
        print("Features:\n" + str(X.loc[original_idx]))
        print("-----------------------------------")
else:
    print("No misclassified samples!")